# Random Forest Regression

A random forest grows many trees, each on a different random sample of the data, and averages their answers. Here it predicts **mpg**, first from **horsepower** alone and then from several inputs.

This notebook goes with the **Random Forest Regression** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

**The steps:** load the data → split → fit → count the trees → try more inputs.

## 1. Load the data

Each row is a car from the late 1970s and early 1980s. We predict `mpg`. A few cars have no horsepower listed, so we drop them first.

The data is the [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) data set from the UCI Machine Learning Repository (CC BY 4.0). See `datasets/README.md` for what we changed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/mpg.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

In [ ]:
dataset = dataset.dropna(subset=['horsepower'])
X = dataset[['horsepower']].values
y = dataset['mpg'].values
print(X.shape, 'cars')

## 2. Split into train and test

The model learns from the **training** cars (75%). We hold back the other 25% and only use them at the end. The test score shows how the model does on cars it has never seen. These are the ringed points on the site's boards.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), 'train,', len(X_test), 'test')

## 3. Fit the forest

`n_estimators` is the number of trees. Every tree sees a bootstrap sample (a random draw with repeats) of the training cars. The forest predicts the **average** of its trees. No scaling is needed.

In [ ]:
regressor = RandomForestRegressor(n_estimators=100, random_state=0)
regressor.fit(X_train, y_train)

## 4. Check the result

**R²** says how much of the ups and downs in the data the model explains. 1 is perfect, 0 is no better than always guessing the average. The number that matters is the **test** R². The last line shows the typical miss in miles per gallon.

In [ ]:
y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)
print('Train R²:', round(r2_score(y_train, y_pred_train), 3))
print('Test R²: ', round(r2_score(y_test, y_pred_test), 3))
print('Test error on average:', round(mean_absolute_error(y_test, y_pred_test), 2), 'mpg')

## 5. How many trees?

One tree is jumpy. Averaging more trees calms it down, until adding more stops helping.

In [ ]:
for n in [1, 3, 10, 30, 100, 300]:
    m = RandomForestRegressor(n_estimators=n, random_state=0).fit(X_train, y_train)
    print(f'{n:>3} trees: test R² {r2_score(y_test, m.predict(X_test)):.3f}')

In [ ]:
grid = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)
plt.figure(figsize=(7, 4.5))
plt.scatter(X_train, y_train, color='#5b7cfa', label='train')
plt.scatter(X_test, y_test, facecolors='white', edgecolors='#5b7cfa', linewidths=1.8, label='test')
plt.plot(grid, regressor.predict(grid), color='black', label='forest (100 trees)')
plt.xlabel('horsepower'); plt.ylabel('mpg'); plt.legend()
plt.show()

## 6. More than one input

A forest handles several inputs without any extra work. Here it uses four, and tells us which ones it relied on most (`feature_importances_`).

In [ ]:
features = ['weight', 'horsepower', 'displacement', 'acceleration']
X4_train, X4_test, y4_train, y4_test = train_test_split(dataset[features].values, y, test_size=0.25, random_state=0)
forest4 = RandomForestRegressor(n_estimators=100, random_state=0).fit(X4_train, y4_train)
print('Test R² with four inputs:', round(r2_score(y4_test, forest4.predict(X4_test)), 3))
print(pd.Series(forest4.feature_importances_, index=features).sort_values(ascending=False).round(3))

## Try this

1. Set `max_depth=3` in the forest. What happens to train and test R²?
2. Ask the forest about a horsepower far outside the data, such as 400. Does the answer keep falling the way a line would?
3. Which input matters most in the four-input forest? Does that match your guess?